In [1]:
# -------------------------------------------------------------
# Common pre‑amble – read config & export core variables
# -------------------------------------------------------------

from pathlib import Path
import os, sys

# infer repo root from the location of this file
repo_root = os.path.abspath('..')
sys.path.insert(0, str(repo_root))  # allow `import src.*`
from config.notebook_setup import *

Repository Root: /home/marcmaceira/projects/reuters-rag-classifier
Configuration: {'run_name': 'experiment_with_10_classes', 'seed': 42, 'n_classes': 10, 'cutoff_year': 1996, 'data_exploration_dir': 'experiment_with_10_classes/data_exploration', 'artifacts_dir': 'experiment_with_10_classes/artifacts', 'embeddings_dir': 'experiment_with_10_classes/embeddings', 'models_dir': 'experiment_with_10_classes/models', 'results_dir': 'experiment_with_10_classes/results', 'embedding_backend': 'sbert', 'sbert_model_name': 'sentence-transformers/all-MiniLM-L12-v2', 'openai_model_name': 'text-embedding-3-small', 'classifier': 'linear_svm', 'rag_top_k': 5, 'use_llm_refine': False, 'batch_size': 32, 'max_epochs': 10, 'learning_rate': '1e-3', 'weight_decay': '1e-4', 'early_stopping': 3, 'test_split': 0.2, 'metrics': ['accuracy', 'macro_f1', 'auc_ovr']}
DATA_EXPLORATION_DIR: /home/marcmaceira/projects/reuters-rag-classifier/experiment_with_10_classes/data_exploration
ARTIFACTS_DIR: /home/marcmaceira/pro

# Reuters News Topic Classification - Exploratory Data Analysis

This notebook performs detailed exploratory data analysis on the Reuters news topic classification dataset. We'll analyze various aspects of the data to better understand its characteristics and potential challenges.

## Setup and Data Loading

In [2]:
import logging
import warnings
import random
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from src.datasets.dataset import load_data
from src.exploration import class_frequency, length_distribution



warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')


# Load dataset
X_train, y_train, X_test, y_test, label_names = load_data(N_CLASSES)
print(f'Train docs: {len(X_train):,},  Test docs: {len(X_test):,}')
print('Labels:', label_names)

Train docs: 6,337,  Test docs: 2,477
Labels: ['earn', 'acq', 'crude', 'interest', 'money-fx', 'trade', 'grain', 'corn', 'dlr', 'money-supply']


## 1. Class Distribution Analysis

Let's analyze the distribution of topics in our dataset to understand class imbalance.

In [3]:
import os

# Ensure the directory exists
os.makedirs("experiment_with_10_classes/vocab_analysis", exist_ok=True)

# Plot length distribution
plt.figure(figsize=(10, 6))
stats = length_distribution(X_train, save_path=os.path.join(DATA_EXPLORATION_DIR, "document_length_distribution.png"))
print(f"Mean length: {stats['stats']['mean']:.1f} tokens, median: {stats['stats']['median']}")

# Calculate length percentiles
lengths = [len(doc.split()) for doc in X_train]
percentiles = np.percentile(lengths, [25, 50, 75, 90, 95, 99])
print("\nDocument Length Percentiles:")
length_stats = []
for p, percentile in zip([25, 50, 75, 90, 95, 99], percentiles):
    print(f"{p}th percentile: {percentile:.0f} tokens")
    length_stats.append({'percentile': p, 'length': int(percentile)})

# Save document length statistics to CSV
pd.DataFrame(length_stats).to_csv(os.path.join(DATA_EXPLORATION_DIR, "document_length_stats.csv"), index=False)

Mean length: 119.9 tokens, median: 77.0

Document Length Percentiles:
25th percentile: 40 tokens
50th percentile: 77 tokens
75th percentile: 142 tokens
90th percentile: 278 tokens
95th percentile: 404 tokens
99th percentile: 667 tokens


<Figure size 1000x600 with 0 Axes>

## 2. Document Length Analysis

Understanding the length distribution of articles helps us make decisions about preprocessing and model architecture.

In [4]:
# Plot length distribution
plt.figure(figsize=(10, 6))


from src.exploration import class_frequency, length_distribution
import pandas as pd


# Plot length distribution
plt.figure(figsize=(10, 6))
stats = length_distribution(X_train, save_path=os.path.join(DATA_EXPLORATION_DIR, "document_length_distribution.png"))
print(f"Mean length: {stats['stats']['mean']:.1f} tokens, median: {stats['stats']['median']}")

# Calculate length percentiles
lengths = [len(doc.split()) for doc in X_train]
percentiles = np.percentile(lengths, [25, 50, 75, 90, 95, 99])
print("\nDocument Length Percentiles:")
length_stats = []
for p, percentile in zip([25, 50, 75, 90, 95, 99], percentiles):
    print(f"{p}th percentile: {percentile:.0f} tokens")
    length_stats.append({'percentile': p, 'length': int(percentile)})

# Save document length statistics to CSV
pd.DataFrame(length_stats).to_csv(os.path.join(DATA_EXPLORATION_DIR, "document_length_stats.csv"), index=False)

Mean length: 119.9 tokens, median: 77.0

Document Length Percentiles:
25th percentile: 40 tokens
50th percentile: 77 tokens
75th percentile: 142 tokens
90th percentile: 278 tokens
95th percentile: 404 tokens
99th percentile: 667 tokens


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

## 3. Vocabulary Analysis

Let's examine the vocabulary characteristics of our dataset.

In [5]:
#!/usr/bin/env python
"""
Simple script to run comprehensive vocabulary analysis on Reuters data.
"""
from src.vocabulary_analysis import comprehensive_analysis
from src.datasets.dataset import load_data

# Load Reuters dataset
print("Loading Reuters dataset...")
X_train, y_train, X_test, y_test, label_names = load_data(10)  # Use top 10 classes
print(f"Loaded {len(X_train)} training documents across {len(label_names)} classes")

# Run comprehensive analysis with custom settings
analysis_results = comprehensive_analysis(
    texts=X_train,                                          # Your text documents
    labels=y_train,                                         # Class labels for class-specific analysis
    label_names=label_names,                                # Names of the classes 
    output_dir=os.path.join(DATA_EXPLORATION_DIR),                    # Output directory for results
    min_word_length=3,                                      # Minimum word length (filters short tokens)
    top_n=50,                                               # Number of top words to analyze
    create_visualizations=True,                             # Create and save plots
    create_csv=True                                         # Save results to CSV files
)

# The analysis_results dictionary contains all the results:
# - analysis_results['basic'] - basic analysis (only stopwords removed)
# - analysis_results['standard'] - standard filtering (stopwords, numbers, financial terms)
# - analysis_results['advanced'] - advanced filtering (adds domain-specific stopwords)
# - analysis_results['basic_class'] - class-specific words with basic filtering
# - analysis_results['standard_class'] - class-specific words with standard filtering
# - analysis_results['advanced_class'] - class-specific words with advanced filtering

print("\nAnalysis complete! Results saved to the 'vocab_analysis_custom' directory.")
print("You can also access the results programmatically through the returned dictionary.")

# Example: Get the top 5 words after advanced filtering
print("\nTop 5 words (advanced filtering):")
for word, count in analysis_results['advanced']['word_counts'].most_common(5):
    print(f"  {word}: {count:,}")

# Example: Get the most distinctive word for each class
print("\nMost distinctive word by class (advanced filtering):")
for class_name in label_names:
    if class_name in analysis_results['advanced_class'].columns:
        top_word = analysis_results['advanced_class'][class_name][0]
        if top_word:  # Check that it's not an empty string
            print(f"  {class_name}: {top_word}") 

Loading Reuters dataset...
Loaded 6337 training documents across 10 classes


NameError: name 'output_dir' is not defined

## 4. Topic Co-occurrence Analysis

Let's examine if there are any patterns in how topics appear together in articles.

In [ ]:
# Create co-occurrence matrix
def get_topic_cooccurrence(labels):
    unique_labels = sorted(set(labels))
    cooccurrence = np.zeros((len(unique_labels), len(unique_labels)))
    
    for i, label1 in enumerate(unique_labels):
        for j, label2 in enumerate(unique_labels):
            if i <= j:  # Only compute upper triangle
                # For a single-label dataset like Reuters, co-occurrence only happens 
                # when i==j (same label)
                if i == j:
                    cooccurrence[i, j] = np.sum(np.array(labels) == label1)
                else:
                    cooccurrence[i, j] = 0
                cooccurrence[j, i] = cooccurrence[i, j]  # Mirror
    
    return pd.DataFrame(cooccurrence, index=unique_labels, columns=unique_labels)

cooccurrence_df = get_topic_cooccurrence(y_train)

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cooccurrence_df, annot=True, fmt='.0f', cmap='YlOrRd')
plt.title('Topic Co-occurrence Matrix')
plt.tight_layout()

plt.savefig(os.path.join(output_dir, "topic_cooccurrence.png"))
cooccurrence_df.to_csv(os.path.join(output_dir, "topic_cooccurrence_stats.csv", index=True))

## Key Findings

1. **Class Imbalance**: The dataset shows significant class imbalance, with 'earn' and 'acq' being the dominant classes.

2. **Document Length**: Most articles are relatively short, with a median length of around 77 tokens. However, there's a long tail of longer articles.

3. **Vocabulary**: The dataset has a rich vocabulary, with many domain-specific financial terms.

4. **Topic Relationships**: Some topics show stronger co-occurrence patterns than others, which could be useful for multi-label classification.

## Next Steps

Based on this analysis, in the next notebook we'll:
1. Implement appropriate class balancing techniques
2. Design a preprocessing pipeline that handles the document length distribution
3. Create an effective vocabulary management strategy